# అధ్యాయం 7: చాట్ అప్లికేషన్ల నిర్మాణం
## మైక్రోసాఫ్ట్ ఫౌండ్రీ మోడల్స్ API త్వరితప్రారంభం

ఈ నోట్బుక్ [ఆజూర్ ఓపెన్‌ఎఐ样品 రిపాజిటరీ](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) నుండి అనుకూలీకరించబడింది, ఇందులో [ఆజూర్ ఓపెన్‌ఎఐ](notebook-azure-openai.ipynb) సేవలను యాక్సెస్ చేసే నోట్బుక్స్ ఉంటాయి.

> **గమనిక:** GitHub మోడల్స్ జూలై 2026 ముగింపున గానీ విరమణ పొందుతున్నాయి. ఈ నోట్బుక్ ఇప్పుడు అదే ఉచిత ప్రయోగం కోసం మోడల్ క్యాటలాగ్ మరియు ఆజూర్ AI ఇన్ఫరెన్స్ SDK అనుభవాన్ని అందించే [మైక్రోసాఫ్ట్ ఫౌండ్రీ మోడల్స్](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) ని లక్ష్యం చేస్తుంది.


# అవలోకనం  
"పెద్ద భాషా నమూనాలు టెక్స్ట్ నుండి టెక్స్ట్ కు మ్యాపింగ్ చేసే ఫంక్షన్లు. ఇచ్చిన ప్రవేశ టెక్స్ట్ స్ట్రింగ్ ఆధారంగా, పెద్ద భాషా నమూనా తరువాత వచ్చే టెక్స్ట్ ను ఊహించడానికి ప్రయత్నిస్తుంది"(1). ఈ "క్విక్‌స్టార్ట్" నోట్‌బుక్ యూజర్లకు ఉన్నతస్థాయి LLM కాన్సెప్ట్స్, AML తో ప్రారంభించడానికి కీలక ప్యాకేజీ అవసరాలు, ప్రాంప్ట్ డిజైన్ కు సాఫ్ట్ పరిచయం, మరియు వివిధ వినియోగాల చిన్న ఉదాహరణలను పరిచయం చేస్తుంది.  


## విషయాలు పట్టిక  

[సారాంశం](#overview)  
[OpenAI సేవను ఎలా ఉపయోగించాలి](#how-to-use-openai-service)  
[1. మీ OpenAI సేవను సృష్టించడం](#1.-creating-your-openai-service)  
[2. సంస్థాపనం](#2.-installation)    
[3. ప్రమాణాలను](#3.-credentials)  

[ఉపయోగ కేసులు](#use-cases)    
[1. వచనం సారాంశం చేయండి](#1.-summarize-text)  
[2. వచనాన్ని వర్గీకరించండి](#2.-classify-text)  
[3. కొత్త ఉత్పత్తి పేర్లను సృష్టించండి](#3.-generate-new-product-names)  
[4. ఒక వర్గీకర్తను_FINE_ చేసుకోండి](#4.fine-tune-a-classifier)  

[సూచనలు](#references)


### మీ మొదటి ప్రాంప్ట్‌ను నిర్మించండి  
ఈ చిన్న వ్యాయామం "సంక్షిప్తీకరణ" అనే సాదా పనికి మైక్రోసాఫ్ట్ ఫౌండ్రి మోడల్స్‌లో మోడల్‌కు ప్రాంప్ట్స్ సమర్పించే ప్రాథమిక పరిచయాన్ని అందిస్తుంది.


**దశలు**:  
1. మీరు ఇప్పటికే చేయకపోతే, మీ పైథాన్ వాతావరణంలో `azure-ai-inference` లైబ్రరీని ఇన్‌స్టాల్ చేయండి.  
2. ప్రామాణిక హెల్పర్ లైబ్రరీస్‌ను లోడ్ చేసి, మీ మైక్రోసాఫ్ట్ ఫౌండ్రి మోడల్స్ క్రెడెన్షియల్స్‌ను సెట్ చేయండి.  
3. మీ పనికి ఒక మోడల్‌ను ఎంచుకోండి  
4. మోడల్ కోసం ఒక సాదా ప్రాంప్ట్ సృష్టించండి  
5. మీ అభ్యర్థనను మోడల్ API కి సమర్పించండి!


### 1. `azure-ai-inference` ను ఇన్‌స్టాల్ చేయండి


In [ ]:
%pip install azure-ai-inference

### 2. సహాయక గ్రంథాలయాలను దిగుమతి చేయండి మరియు ప్రమాణాల‌ను సృష్టించండి


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. సరైన మోడల్‌ను కనుగొనడం  
GPT-4o మరియు GPT-4o మిని వంటి మోడల్స్ సహజ భాషను అర్థం చేసుకొని ఉత్పత్తి చేయగలవు, మరియు ఇవి Microsoft Foundry Models క్యాటలాగ్‌లో Meta, Mistral, Cohere, మరియు Microsoft నుండి వచ్చిన మోడల్స్ తో పాటు అందుబాటులో ఉన్నాయి.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. ప్రాంప్ట్ ವಿನ್ಯಾಸం  

"విశాల వచనాలపై ఈ అంచనా తప్పును తక్కువ చేసేందుకు శిక్షణ పొందటం ద్వారా, పెద్ద భాషామోడల్స్ ఈ అంచనాలకు ఉపయోగపడే భావనలను నేర్చుకుంటాయి అనే వింత విషయం ఉంది. ఉదాహరణకు, ఇవి ఈ విధంగా భావనలను నేర్చుకుంటాయి"(1):

* ఎలాగ చెప్పాలి
* వ్యాకరణం ఎలా పనిచేస్తుందో
* ఎలా ప్యారాఫ్రేస్ చెయ్యాలి
* ప్రశ్నలకు ఎలా సమాధానం ఇవ్వాలి
* సంభాషణ ఎలా జరగాలి
* అనేక భాషల్లో ఎలా వ్రాయాలి
* కోడింగ్ ఎలా చేసుకోవాలి
* తదితరాలు

#### పెద్ద భాషా మోడల్‌ను ఎలా నియంత్రించాలి  
"పెద్ద భాషామోడల్‌కు అన్ని ఇన్‌పుట్లు ఉన్నా, అత్యంత ప్రభావవంతమైనది టెక్స్ట్ ప్రాంప్ట్(1).

పెద్ద భాషామోడల్స్‌ను ఉత్పత్తి చేయమని ప్రాంప్ట్ చేయడానికి కొన్ని మార్గాలు ఉన్నాయి:

సూచన: మీరు కావలసింది మోడల్‌కు చెప్పండి
పూర్తి చేయడం: మీరు కావలసినదాన్ని మొదలు పెట్టి మోడల్‌ పూర్తిచేయమని కోరండి
ప్రదర్శన: మీరు కావలసినదాన్ని మోడల్‌కు చూపండి, క్రింద లేదా:
ప్రాంప్ట్ లో కొద్ది ఉదాహరణలు
మంచి శిక్షణ డేటాసెట్‌లో వందల లేక వేల ఉదాహరణలు"



#### ప్రాంప్ట్ సృష్టించడానికి మూడు ప్రాథమిక మార్గనిర్దేశాలు ఉన్నాయి:

**చూపించండి మరియు చెప్పండి**. సూచనల ద్వారా, ఉదాహరణల ద్వారా లేదా రెండింటి సంయోగం ద్వారా మీరు ఏమి కావాలో స్పష్టం చేయండి. మీరు మోడల్‌ను అక్షరాల అమరికలో అంశాలను వరుసగా చేసి ర్యాంకు చేయమని లేదా ఒక పేరాగ్రాఫ్‌ను భావ ప్రకటన ద్వారా వర్గీకరించమని కోరితే, అది మీరు కావాలని చూపించండి.

**నాణ్యమైన డేటా ఇచ్చండి**. మీరు క్లాసిఫయర్ లేదా నమూనా అనుసరించి మోడల్‌ను నిర్వహించాలి అనుకుంటే, సరిపడమైన ఉదాహరణలు ఉండాలి. మీ ఉదాహరణలను సరిచూడండి — మోడల్ సాధారణ వ్రుత్తి తప్పులను చూసి సమాధానం ఇవ్వడానికి చురుకైనది కానీ, ఇది చింతించని దృక్కోణం అనుకొని ప్రతిస్పందనపై ప్రభావం చూపవచ్చు.

**మీ సెట్టింగులను పరీక్షించండి.** టెంపరేచర్ మరియు టాప్_పి సెట్టింగులు ప్రతిస్పందన ఉత్పత్తిలో మోడల్ ఎంత డిటర్మినిస్టిక్ అని నియంత్రిస్తాయి. మీరు ఒక్కటి సరైన సమాధానం ఉన్న ప్రశ్న అడుగుతుంటే వీటిని తగ్గించాలి. విభిన్నమైన ప్రత్యుత్తరాలు కావాలనుకుంటే వీటిని పెంచాలి. ఈ సెట్టింగులతో అవకాశం పొరపాటు అనేది అవి "క్లెవర్నెస్" లేదా "సృజనాత్మకత" నియంత్రణలు అని భావించడం.


మూలం: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. సమర్పించండి!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### అదే కాల్‌ని మళ్ళీ చేయండి, ఫలితాలు ఎలా పోల్చుకుంటాయి?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## వచనం సారాంశం చేయండి  
#### సవాలు  
వచనం చివరకు 'tl;dr:' ను జోడించడం ద్వారా వచనం సారాంశం చేయండి. మోడల్ ఎలా అనేక పనులను అదనపు సూచనలు లేకుండానే ఎలా చేయాలో అర్థం చేసుకుంటుందో గమనించండి. మీరు tl;dr కంటే వివరణాత్మక ప్రాంప్ట్‌లతో మోడల్ పనితీరును మార్చి మీరు పొందే సారాంశాన్ని అనుకూలీకరించవచ్చు(3).  

తాజా పనులు విస్తృత సంఖ్యలో వచనంపై ప్రీ-ట్రెయినింగ్ చేసి, తరువాత ప్రత్యేక ప‌నిపై ఫైన్-ట్యూనింగ్ చేయడం ద్వారా అనేక NLP పనులు మరియు బెంచ్‌మార్కులపై గొప్ప పురోగతులను చూపాయి. సాధారణంగా టాస్క్-ఏగ్నాస్టిక్ కోటులో ఉన్నప్పటికీ, ఈ పద్ధతి ఇంకా వేలల యాదృచ్ఛిక విలువలతో కూడిన పనికి ప్రత్యేక ఫైన్-ట్యూనింగ్ డేటాసెట్లు అవసరం. భిన్నంగా, మానవులు సాధారణంగా కొద్ది ఉదాహరణలు లేదా సులభమైన సూచనల నుంచి కొత్త భాష పనిని చేయగలరు - ఇది ప్రస్తుత NLP వ్యవస్థలు చాలా వరకు చేయలేకపోతున్న విషయం. ఇక్కడ మేము చూపిస్తున్నాము భాషా మోడల్స్ ను పెంచడం వలన టాస్క్-ఏగ్నాస్టిక్, కొద్దిసార్లు పనితీరులో గణనీయమైన మెరుగుదల వస్తుందనేది, ఇది తరచుగా మునుపటి అత్యున్నత ఫైన్-ట్యూనింగ్ పద్ధతులతో పోటీ పడగలదు.  



Tl;dr  


# వివిధ ఉపయోగాల కోసం అభ్యాసాలు  
1. పాఠ్యాన్ని సారాంశం చేయండి  
2. పాఠ్యాన్ని వర్గీకరించండి  
3. కొత్త ఉత్పత్తి పేర్లు సృష్టించండి  


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## వర్గీకరణ పాఠ్యం  
#### సవాలు  
దార్మి కాలంలో ఇచ్చిన వర్గాల ప్రకారం అంశాలను వర్గీకరించండి. తరువాత ఉదాహరణలో, మేము వర్గాలు మరియు వర్గీకరించవలసిన పాఠ్యాన్ని కూడా ప్రాంప్ట్(*playground_reference)లో ఇస్తున్నాం. 

కస్టమర్ విచారణ: హలో, నా ల్యాప్టాప్ కీబోర్డ్ లో ఒక కీ ఇటీవలుగా పగిలింది మరియు నాకు దాని ప్రత్యామ్నాయం కావాలి:

వర్గీకరించిన వర్గం:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## కొత్త ఉత్పత్తి పేర్లు సృష్టించండి
#### సవాలు
ఉదాహరణ పదాల నుండి ఉత్పత్తి పేర్లను సృష్టించండి. ఇక్కడ మేము ఉత్పత్తి గురించి సమాచారం_PROMPTలో ఇవ్వడం జరిగింది. మేము ఎలాంటి నమూనా అందుకోవాలని చూపించడానికి ఒక సమాన ఉదాహరణను కూడా ఇస్తాము. అత్యధిక అనూహ్య స్పందనలను పెంచడానికి మేము టెంపరేచర్ విలువను కూడా ఎక్కువగా సెట్ చేశాము.

ఉత్పత్తి వివరణ: హోమ్ మిల్క్‌షేకు తయారీ యంత్రం
మూల పదాలు: వేగవంతమైన, ఆరోగ్యకరమైన, సన్నని.
ఉత్పత్తి పేర్లు: హోమ్‌షేకర్, ఫిట్స్‌షేకర్, క్విక్‌షేక్, షేక్ మేకర్

ఉత్పత్తి వివరణ: ఏ పాద పరిమాణానికైనా సరిపోయే జత జత చెప్పునులు
మూల పదాలు: అనుకూలించు, సరిపో,’omni-fit‘.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# సూచనలు  
- [Openai కుక్‌బుక్](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry పోర్టల్](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [పాఠ్యాన్ని వర్గీకరించడానికి GPT నమూనాలను సరిగ్గా అనుకూలీకరించే ఉత్తమ అమలు పద్ధతులు](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# మరింత సహాయం కోసం  
[OpenAI వాణిజ్యీకరణ బృందం](AzureOpenAITeam@microsoft.com) 


# సహకారులు
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
